## 5. Joins: INNER JOIN, LEFT JOIN, LEFT OUTER JOIN, UNION ALL, UNION, AS, CROSS JOIN, NATURAL JOIN

In this notebook we will cover joins. This is how you connect two tables, or even one to itself.

Not in SQlite but covered RIGHT JOIN, FULL OUTER JOIN, SELF JOIN

###### Keywords Covered - INNER JOIN, LEFT JOIN, LEFT OUTER JOIN, UNION ALL, UNION, AS, CROSS JOIN, NATURAL JOIN

<div align="center"><b>Join Types</b></div>


| Join Type      | Rows Returned                     | Use Case                              |
|:---------------|:----------------------------------|:--------------------------------------|
| INNER JOIN     | Only matches                      | When you need data present in both tables |
| LEFT JOIN      | All left + matching right         | When you want to keep the records from a main table |
| RIGHT JOIN     | All right + matching left         | Ditto above (switch tables and use LEFT JOIN instead)  |
| FULL OUTER     | All from both tables              | When you need all of the data from both tables       |
| CROSS JOIN     | All possible combinations         | Generating combinations              |
| SELF JOIN      | Table joined with itself          | Hierarchical or recursive data         |
| NATURAL JOIN   | Only matches in common columns | Considered bad practise - Don't use |

In [1]:
import sqlite3
import pandas as pd

# Connect to database
data_conn = sqlite3.connect("pokemon.db")

First we will build some demonstration tables for these JOIN examples.

In [2]:
# Drop the demo tables created in this workbook.
data_conn.execute("DROP TABLE pokemon_item_price")
data_conn.execute("DROP TABLE pokemon_items")
data_conn.execute("DROP TABLE pokemon_trainers") 

In [3]:
# Create the tables for the Pokémon items and their prices.
data_conn.execute("CREATE TABLE pokemon_item_price \
                        (item_id INTEGER PRIMARY KEY AUTOINCREMENT, \
                         name TEXT NOT NULL, \
                         price INTEGER NOT NULL)")

data_conn.execute("CREATE TABLE pokemon_items \
                        (name TEXT NOT NULL, \
                         effect TEXT NOT NULL)")

In [4]:
# Insert data into the two tables.
data_conn.execute("INSERT INTO pokemon_item_price (name, price) VALUES \
                ('Poké Ball', 200), ('Great Ball', 600), \
                ('Potion', 300), ('Super Potion', 700), ('Hyper Potion', 1200), \
                ('Revive', 1500), ('Escape Rope', 550), ('Antidote', 100), ('Burn Heal', 250), ('Ice Heal', 250)")

data_conn.execute("INSERT INTO pokemon_items (name, effect) VALUES \
                ('Poké Ball', 'Catches Pokémon.'), ('Great Ball', 'Catches Pokémon better.'), ('Ultra Ball', 'Catches Pokémon great!'), \
                ('Potion', 'Heals 20 HP.'), ('Super Potion', 'Heals 50 HP.'), ('Full Restore', 'Fully restores HP and cures all effects.'), \
                ('Revive', 'Revives your Pokémon.'), ('Escape Rope', 'Takes you to the entrance of a cave.')")

In [5]:
pkmn_item_price = data_conn.execute("SELECT * FROM pokemon_item_price").fetchall()
df = pd.DataFrame(pkmn_item_price) # Convert list to dataframe.
df.columns= ['Id', 'Name', 'Price'] # Add column names.
df

,Id,Name,Price
0,1,Poké Ball,200
1,2,Great Ball,600
2,3,Potion,300
3,4,Super Potion,700
4,5,Hyper Potion,1200
5,6,Revive,1500
6,7,Escape Rope,550
7,8,Antidote,100
8,9,Burn Heal,250
9,10,Ice Heal,250


In [6]:
pkmn_item_price = data_conn.execute("SELECT * FROM pokemon_items").fetchall()
df = pd.DataFrame(pkmn_item_price) # Convert list to dataframe.
df.columns= ['Name', 'Effect'] # Add column names.
df

,Name,Effect
0,Poké Ball,Catches Pokémon.
1,Great Ball,Catches Pokémon better.
2,Ultra Ball,Catches Pokémon great!
3,Potion,Heals 20 HP.
4,Super Potion,Heals 50 HP.
5,Full Restore,Fully restores HP and cures all effects.
6,Revive,Revives your Pokémon.
7,Escape Rope,Takes you to the entrance of a cave.


#### INNER JOIN

Use: Joins two tables, but returns only rows where there is a match in both of the tables.

*Tip: Use abreviated table names by putting an abreviation after the table being joined, such as c fors costs.*

We will use INNER JOIN on our pokemon_items and pokemon_item_price tables below.

In [7]:
inner_j = data_conn.execute("SELECT costs.item_id, costs.name, costs.price, items.effect \
                             FROM pokemon_items items \
                             INNER JOIN pokemon_item_price costs \
                             ON items.name = costs.name").fetchall()
df = pd.DataFrame(inner_j)
df.columns= ['Id', 'Name', 'Cost', 'Effect'] # Add column names.
df

,Id,Name,Cost,Effect
0,1,Poké Ball,200,Catches Pokémon.
1,2,Great Ball,600,Catches Pokémon better.
2,3,Potion,300,Heals 20 HP.
3,4,Super Potion,700,Heals 50 HP.
4,6,Revive,1500,Revives your Pokémon.
5,7,Escape Rope,550,Takes you to the entrance of a cave.


#### LEFT JOIN (LEFT OUTER JOIN)

Use: Returns all rows from the left table with matching rows from the right table.
Non-matching rows from the right table get NULL values.

*RIGHT JOIN is not supported in SQlite, but you perform it by swapping the tables and doing a LEFT JOIN.
RIGHT JOIN returns all of the rows from the right tables with matching rows from the left table.*

*Tip: LEFT JOIN and LEFT OUTER JOIN have identical functionality in SQLite.*

We will use JOIN LEFT on our pokemon_items and pokemon_item_price tables below.

pokemon_item_price is the Left table.

pokemon_items is the Right table.

So the items that are in the pokemon_item_price but not in pokemon_items will have NULL in the Effect column.

In [8]:
left_j = data_conn.execute("SELECT costs.item_id, costs.name, costs.price, items.effect \
                             FROM pokemon_item_price costs \
                             LEFT JOIN pokemon_items items \
                             ON items.name = costs.name").fetchall()
df = pd.DataFrame(left_j)
df.columns= ['Id', 'Name', 'Cost', 'Effect'] # Add column names.
df

,Id,Name,Cost,Effect
0,1,Poké Ball,200,Catches Pokémon.
1,2,Great Ball,600,Catches Pokémon better.
2,3,Potion,300,Heals 20 HP.
3,4,Super Potion,700,Heals 50 HP.
4,5,Hyper Potion,1200,NaN
5,6,Revive,1500,Revives your Pokémon.
6,7,Escape Rope,550,Takes you to the entrance of a cave.
7,8,Antidote,100,NaN
8,9,Burn Heal,250,NaN
9,10,Ice Heal,250,NaN


#### FULL OUTER JOIN 

Use: This will combine both tables and return ALL of the rows from both tables.
If a row matches on the join, conditon, they will join, else they will fill with NULL values.

This is not supported in SQLite, so you must use LEFT JOIN and UNION ALL.
You call UNION ALL on effectively a RIGHT JOIN and a LEFT JOIN of two tables, as shown below.

*Tip: Use if you want to keep all of the data from both tables.*

In the example below we will FULL OUTER JOIN the pokemon_item_price (right) and pokemon_items (left) tables.

#### UNION ALL

Use: Combines the results of two SELECT statements.

#### UNION

Use: Combines the results of two SELECT statements, but removes duplicates.

Below NULL AS item_id and NULL AS price are used in the second SELECT LEFT JOIN, as they are not columns in the  pokemon_items table.

#### AS

Use: Create aliases for columns, for instanse SELECT NULL AS item_id creates a column full of NULL values.

You could use this to rename a column in a view for example *SELECT price AS Cost*.

In [9]:
fo_j = data_conn.execute("SELECT costs.item_id, costs.name, costs.price, items.effect \
                          FROM pokemon_item_price costs \
                          LEFT JOIN pokemon_items items \
                          ON items.name = costs.name \
                          \
                          UNION ALL \
                          \
                          SELECT NULL AS item_id, items.name, NULL AS price, items.effect \
                          FROM pokemon_items items \
                          LEFT JOIN pokemon_item_price costs \
                          ON costs.name = items.name \
                          WHERE costs.name IS NULL").fetchall()
df = pd.DataFrame(fo_j)
df.columns= ['Id', 'Name', 'Cost', 'Effect'] # Add column names.
df

,Id,Name,Cost,Effect
0,1.0,Poké Ball,200.0,Catches Pokémon.
1,2.0,Great Ball,600.0,Catches Pokémon better.
2,3.0,Potion,300.0,Heals 20 HP.
3,4.0,Super Potion,700.0,Heals 50 HP.
4,5.0,Hyper Potion,1200.0,NaN
5,6.0,Revive,1500.0,Revives your Pokémon.
6,7.0,Escape Rope,550.0,Takes you to the entrance of a cave.
7,8.0,Antidote,100.0,NaN
8,9.0,Burn Heal,250.0,NaN
9,10.0,Ice Heal,250.0,NaN


In [10]:
# Below NULL AS item_id and NULL AS price are used in the second SELECT LEFT JOIN.
# Below shows the result of the second LEFT JOIN, before it is combined with the first one.
fo_j = data_conn.execute("SELECT NULL AS item_id, items.name, NULL AS price, items.effect \
                          FROM pokemon_items items \
                          LEFT JOIN pokemon_item_price costs \
                          ON costs.name = items.name \
                          WHERE costs.name IS NULL").fetchall()
df = pd.DataFrame(fo_j)
df.columns= ['Id', 'Name', 'Cost', 'Effect'] # Add column names.
df

,Id,Name,Cost,Effect
0,None,Ultra Ball,None,Catches Pokémon great!
1,None,Full Restore,None,Fully restores HP and cures all effects.


#### CROSS JOIN

Use: Returns the Cartesian product. This is every row from the left table comboined with every row from the right table.

In the example below we will cross the pokemon_item_price (right) and pokemon_items (left) tables.

Using the name from costs and effects from items, this will show every combination of name and effect.

In [11]:
cross_j = data_conn.execute("SELECT costs.name, items.effect \
                             FROM pokemon_item_price costs \
                             CROSS JOIN pokemon_items items").fetchall()
df = pd.DataFrame(cross_j)
df.columns= ['Name', 'Effect'] # Add column names.
df

,Name,Effect
0,Poké Ball,Catches Pokémon.
1,Poké Ball,Catches Pokémon better.
2,Poké Ball,Catches Pokémon great!
3,Poké Ball,Heals 20 HP.
4,Poké Ball,Heals 50 HP.
...,...,...
75,Ice Heal,Heals 20 HP.
76,Ice Heal,Heals 50 HP.
77,Ice Heal,Fully restores HP and cures all effects.
78,Ice Heal,Revives your Pokémon.


#### SELF JOIN

Use: Showing relationships within a single table by linking accorss the same key in multiple coumns.

For example belwo we will make atable that has each trainer with thier gym leader's id, then SELF JOIN to show the name of each trainers gym leader.

In [12]:
# Creating example table for hierarchical data.
data_conn.execute("CREATE TABLE pokemon_trainers( \
                        id INTEGER PRIMARY KEY, \
                        name TEXT NOT NULL, \
                        role TEXT NOT NULL,\
                        leader_id INTEGER)")

# Insert the data.
data_conn.execute("INSERT INTO pokemon_trainers (id, name, role, leader_id) VALUES \
            (1, 'Brock', 'Gym Leader', NULL), (2, 'Misty', 'Gym Leader', NULL), \
            (3, 'Lass', 'Trainer', 2), (4, 'Youngster', 'Trainer', 1), \
            (5, 'Junior Trainer', 'Trainer', 1), (6, 'Swimmer', 'Trainer', 2)")

In [13]:
# Display the table.
trainers = data_conn.execute("SELECT * FROM pokemon_trainers").fetchall()

df = pd.DataFrame(trainers)
df.columns= ['id', 'Name', "Role", "Leader_id"] # Add column names.
df

,id,Name,Role,Leader_id
0,1,Brock,Gym Leader,NaN
1,2,Misty,Gym Leader,NaN
2,3,Lass,Trainer,2.0
3,4,Youngster,Trainer,1.0
4,5,Junior Trainer,Trainer,1.0
5,6,Swimmer,Trainer,2.0


In [14]:
# Do the SELF JOIN using t and g as aliases for pokemon_trainers.
sj = data_conn.execute("SELECT t.name as trainer, \
                          t.role, \
                          g.name as gym_leader \
                   FROM pokemon_trainers t \
                   LEFT JOIN pokemon_trainers g \
                      ON t.leader_id = g.id")                          
df = pd.DataFrame(sj)
df.columns= ['Name', "Role", "Gym Leader"] # Add column names.
df

,Name,Role,Gym Leader
0,Brock,Gym Leader,NaN
1,Misty,Gym Leader,NaN
2,Lass,Trainer,Misty
3,Youngster,Trainer,Brock
4,Junior Trainer,Trainer,Brock
5,Swimmer,Trainer,Misty


#### NATURAL JOIN

Use: A quicker version of INNER JOIN that matches ON columns with matching names from two tables.

In the case below it produces the same result as an INNER JOIN ON items.name = costs.name would for the two tables.

In [15]:
natural_j = data_conn.execute("SELECT costs.item_id, costs.name, costs.price, items.effect \
                             FROM pokemon_items items \
                             NATURAL JOIN pokemon_item_price costs").fetchall()
df = pd.DataFrame(natural_j)
df.columns= ['Id', 'Name', 'Cost', 'Effect'] # Add column names.
df

,Id,Name,Cost,Effect
0,1,Poké Ball,200,Catches Pokémon.
1,2,Great Ball,600,Catches Pokémon better.
2,3,Potion,300,Heals 20 HP.
3,4,Super Potion,700,Heals 50 HP.
4,6,Revive,1500,Revives your Pokémon.
5,7,Escape Rope,550,Takes you to the entrance of a cave.
